<a href="https://colab.research.google.com/github/semehal62/aws-sbg-tasks-/blob/main/semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

url = "https://raw.githubusercontent.com/semehal62/aws-sbg-tasks-/main/Week%202/FAQ.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(15, 3)


,id,question,answer
0,1,How do I reset my password?,Click 'Forgot password' on the login page and ...
1,2,How can I change my email address?,"Go to Account Settings, choose Profile, and up..."
2,3,How do I delete my account?,"Open Account Settings, scroll to the bottom, a..."
3,4,What payment methods do you accept?,"We accept credit cards, debit cards, PayPal, a..."
4,5,How do I get a refund?,Contact support within 14 days of purchase wit...


In [13]:
print(df.columns.tolist())
print(df.shape)
df.head()

['id', 'question', 'answer']
(15, 3)


,id,question,answer
0,1,How do I reset my password?,Click 'Forgot password' on the login page and ...
1,2,How can I change my email address?,"Go to Account Settings, choose Profile, and up..."
2,3,How do I delete my account?,"Open Account Settings, scroll to the bottom, a..."
3,4,What payment methods do you accept?,"We accept credit cards, debit cards, PayPal, a..."
4,5,How do I get a refund?,Contact support within 14 days of purchase wit...


In [14]:
QUESTION_COL = "question"   # change to your question column name
ANSWER_COL = "answer"       # change to your answer column name

# Quick check: should print True
print(QUESTION_COL in df.columns and ANSWER_COL in df.columns)

True


# Clean the Data


In [15]:
import re

df = df.dropna(subset=[QUESTION_COL, ANSWER_COL])
df = df.drop_duplicates(subset=[QUESTION_COL]).reset_index(drop=True)

def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df[QUESTION_COL] = df[QUESTION_COL].apply(clean_text)
df[ANSWER_COL] = df[ANSWER_COL].apply(clean_text)

df["text"] = "Q: " + df[QUESTION_COL] + " A: " + df[ANSWER_COL]
df["id"] = range(1, len(df) + 1)   # fresh ids, in case your file has none

print(df.shape)
df[["id", "text"]].head()

(15, 4)


,id,text
0,1,Q: How do I reset my password? A: Click 'Forgo...
1,2,Q: How can I change my email address? A: Go to...
2,3,Q: How do I delete my account? A: Open Account...
3,4,Q: What payment methods do you accept? A: We a...
4,5,Q: How do I get a refund? A: Contact support w...


# Creating Chuck function

In [16]:
def chunk_text(text, max_words=100, overlap=20):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

print(len(chunk_text("word " * 250)))

3


#  Chunk every FAQ

In [17]:
rows = []
for _, row in df.iterrows():
    for i, chunk in enumerate(chunk_text(row["text"])):
        rows.append({
            "chunk_id": f"{row['id']}_{i}",
            "faq_id": row["id"],
            "question": row[QUESTION_COL],
            "text": chunk
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.shape)
chunks_df.head()

(15, 4)


,chunk_id,faq_id,question,text
0,1_0,1,How do I reset my password?,Q: How do I reset my password? A: Click 'Forgo...
1,2_0,2,How can I change my email address?,Q: How can I change my email address? A: Go to...
2,3_0,3,How do I delete my account?,Q: How do I delete my account? A: Open Account...
3,4_0,4,What payment methods do you accept?,Q: What payment methods do you accept? A: We a...
4,5_0,5,How do I get a refund?,Q: How do I get a refund? A: Contact support w...


# Install the libraries

In [18]:
!pip install -q chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


# check that chuck exits

In [19]:
print(chunks_df.shape)
chunks_df.head()

(15, 4)


,chunk_id,faq_id,question,text
0,1_0,1,How do I reset my password?,Q: How do I reset my password? A: Click 'Forgo...
1,2_0,2,How can I change my email address?,Q: How can I change my email address? A: Go to...
2,3_0,3,How do I delete my account?,Q: How do I delete my account? A: Open Account...
3,4_0,4,What payment methods do you accept?,Q: What payment methods do you accept? A: We a...
4,5_0,5,How do I get a refund?,Q: How do I get a refund? A: Contact support w...


# Load the embedding model


In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Turn every chunk into an embedding

In [21]:
embeddings = model.encode(chunks_df["text"].tolist(), show_progress_bar=True)
print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(15, 384)


# Store them in Chroma


In [22]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(
    name="faqs",
    metadata={"hnsw:space": "cosine"}   # measure similarity by meaning/direction
)

collection.upsert(
    ids=chunks_df["chunk_id"].astype(str).tolist(),
    documents=chunks_df["text"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {"faq_id": int(fid), "question": q}
        for fid, q in zip(chunks_df["faq_id"], chunks_df["question"])
    ]
)

print(collection.count())

15


In [23]:
print(collection.count())

15


# search Function


In [24]:
def search(query, n_results=3):
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    print(f"Query: {query}\n")
    for i in range(len(results["ids"][0])):
        distance = results["distances"][0][i]
        similarity = 1 - distance          # cosine: higher = more similar
        meta = results["metadatas"][0][i]
        print(f"Match {i+1}  (similarity: {similarity:.2f})")
        print(f"  FAQ question: {meta['question']}")
        print(f"  Text: {results['documents'][0][i]}")
        print()

In [25]:
search("I forgot my login details")

Query: I forgot my login details

Match 1  (similarity: 0.58)
  FAQ question: How do I reset my password?
  Text: Q: How do I reset my password? A: Click 'Forgot password' on the login page and follow the email link to set a new one.

Match 2  (similarity: 0.42)
  FAQ question: How do I delete my account?
  Text: Q: How do I delete my account? A: Open Account Settings, scroll to the bottom, and click Delete Account. This cannot be undone.

Match 3  (similarity: 0.32)
  FAQ question: How can I change my email address?
  Text: Q: How can I change my email address? A: Go to Account Settings, choose Profile, and update your email. You will need to verify the new address.

